# Detector playground

Practice with the four functions in `training_data/detectors.py` using
Noah's planar 4.8.8 circuits. Basic Python and NumPy indexing are enough to start.

Open this notebook after following the repository setup instructions.
Select your **Color code training data** kernel, then choose **Run All**,
or use **Shift+Enter** to run one cell at a time.
The notebook uses the CPU and small samples.

1. Try measurement masks.
2. Build a circuit and inspect detector descriptions.
3. Sample measurements and check your parities against Stim.
4. Organize detector columns and check a noiseless circuit.

The first four sections cover detector inputs and logical-flip labels.
Section 5 inspects the new adapted local correction targets. A logical-flip label of 1
alone does not mean a decoder failed; no decoder is running here.

In [ ]:
from pathlib import Path
import sys

current_directory = Path.cwd()
candidate_directories = [current_directory]
candidate_directories.extend(current_directory.parents)
child_directory = current_directory / "color_code_training_data"
candidate_directories.append(child_directory)

project_root = None
for candidate_directory in candidate_directories:
    learning_file = candidate_directory / "training_data" / "detectors.py"
    if learning_file.is_file():
        project_root = candidate_directory
        break

if project_root is None:
    raise RuntimeError("Open the repository directory first.")

project_path = str(project_root)
if project_path not in sys.path:
    sys.path.insert(0, project_path)

print("Python:", sys.executable)
print("Project:", project_root)

## Load your code

Autoreload picks up saved edits to your Python file before the next cell runs.
After an edit, rerun the cells that create the values you want to inspect.
After changing experiment parameters, **Restart Kernel** and **Run All** gives
you a fresh, consistent experiment.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import stim
from pprint import pprint
from training_data import detectors as learning
from training_data.planar_488.dependencies import load_noah_builder

noah_builder = load_noah_builder()

print("Learning file:", learning.__file__)
print("NumPy:", np.__version__)
print("Stim:", stim.__version__)

## 1. Try a measurement mask

Count bit positions from the right, starting at zero. Predict the positions
selected by `0b10110` before running this cell. Change the mask and try again.

In [ ]:
example_mask = 0b10110
selected_positions = learning.record_indices(example_mask)
binary_mask = bin(example_mask)

print("Mask:", binary_mask)
print("Selected positions:", selected_positions)

assert learning.record_indices(0) == []
assert learning.record_indices(0b10110) == [1, 2, 4]

## 2. Choose a small experiment

Start with distance 3. Valid planar 4.8.8 distances include 3, 7, and 11.
`rounds` counts measurement cycles. `basis` is the memory basis, X or Z.
`start_phase` chooses which of the two alternating circuit patterns goes first.
`noise_probability` is the source SI1000 noise parameter, rather than a
uniform probability for every circuit operation. Samples stay in memory.

In [ ]:
distance = 3
rounds = 3
basis = "Z"
start_phase = 0
noise_probability = 0.001
shot_count = 256
seed = 42

In [ ]:
geometry, cycle_circuits, cycle_maps = noah_builder._triangle_components(distance, "native-cz")

circuit = noah_builder.build_memory(
    "planar_488",
    distance,
    basis,
    rounds=rounds,
    start_phase=start_phase,
    gate_set="native-cz",
    p=noise_probability,
)

faces = geometry["plaquettes"]
face_count = len(faces)
ancilla_coordinates = geometry["ancilla_coords"]
ancilla_count = len(ancilla_coordinates)

print("Faces:", face_count)
print("Ancillas:", ancilla_count)
print("Measurements per shot:", circuit.num_measurements)
print("Detectors:", circuit.num_detectors)
print("Logical observables:", circuit.num_observables)

### Describe the detectors

Each dictionary identifies a detector and the measurement positions it uses.
`measurement_support` contains positions in the complete record. Constraint
descriptions also include positions within a single round.

In [ ]:
descriptors = learning.describe_detectors(
    geometry,
    cycle_maps,
    rounds,
    basis,
    start_phase,
)

descriptor_count = len(descriptors)
assert descriptor_count == circuit.num_detectors

first_descriptor = descriptors[0]
final_descriptor = descriptors[-1]

print("First detector:")
pprint(first_descriptor)
print("Final detector:")
pprint(final_descriptor)

## 3. Sample measurement outcomes

One row is one shot, meaning one run of the complete circuit. One column is
one measurement position. The noiseless reference fixes the baseline for
detector events and logical flips; individual noiseless measurements can
still be random.

In [ ]:
clean_circuit = circuit.without_noise()
reference_measurements = clean_circuit.reference_sample()

sampler = circuit.compile_sampler(seed=seed)
measurements = sampler.sample(shot_count)

first_measurement_row = measurements[0]
readable_measurements = first_measurement_row.astype(int)

print("Measurement array shape:", measurements.shape)
print("First shot:", readable_measurements)

### Check your parity calculation against Stim

Your function XORs the selected measurements relative to the reference.
Stim's measurement-to-detector converter provides an independent comparison
for both detector events and logical-flip labels. Any mismatch raises an error.

In [ ]:
detector_events, logical_flips = learning.measurement_parities(
    circuit,
    measurements,
    reference_measurements,
    descriptors,
)

converter = circuit.compile_m2d_converter()
stim_detector_events, stim_logical_flips = converter.convert(
    measurements=measurements,
    separate_observables=True,
)

np.testing.assert_array_equal(detector_events, stim_detector_events)
np.testing.assert_array_equal(logical_flips, stim_logical_flips)

print("Parity checks passed against Stim.")
print("Detector array shape:", detector_events.shape)
print("Logical label shape:", logical_flips.shape)

### Work through one detector by hand

Change `shot_index` and `detector_index` to inspect another example.
Predict the XOR of `differences` before looking at `manual_parity`.

In [ ]:
shot_index = 0
detector_index = 0

descriptor = descriptors[detector_index]
measurement_positions = descriptor["measurement_support"]
selected_measurements = measurements[shot_index, measurement_positions]
selected_reference = reference_measurements[measurement_positions]
differences = selected_measurements ^ selected_reference
manual_parity = np.logical_xor.reduce(differences)
computed_parity = detector_events[shot_index, detector_index]

pprint(descriptor)
print("Selected outcomes:", selected_measurements)
print("Reference outcomes:", selected_reference)
print("Differences:", differences)
print("Manual parity:", manual_parity)
print("Function parity:", computed_parity)

assert manual_parity == computed_parity

## 4. Organize the detector columns

Face events use axes `(shot, X/Z, round or final boundary, face)`.
X is basis index 0 and Z is index 1. Flag events use
`(shot, round, constraint)`. The presence mask describes which face checks
exist, independently of whether any detection event happened.

In [ ]:
first_phase_map = cycle_maps[0]
record_constraints = first_phase_map["record_constraints"]
constraint_count = len(record_constraints)

structured_inputs = learning.structure_detectors(
    detector_events,
    descriptors,
    rounds,
    face_count,
    constraint_count,
)

face_events = structured_inputs.face_detection_events
face_present = structured_inputs.face_present
flag_events = structured_inputs.flag_detection_events

print("Face events:", face_events.shape)
print("Face presence:", face_present.shape)
print("Flag events:", flag_events.shape)

### Verify that every detector column reached its intended slot

This checks the original values against their locations in the organized
arrays, including the final boundary and both check bases where present.

In [ ]:
for detector_index, descriptor in enumerate(descriptors):
    round_index = descriptor["round"]
    detector_kind = descriptor["kind"]

    if detector_kind == "face":
        check_basis = descriptor["basis"]
        if check_basis == "X":
            basis_index = 0
        else:
            basis_index = 1

        face_index = descriptor["face"]
        organized_column = face_events[:, basis_index, round_index, face_index]
        assert face_present[basis_index, round_index, face_index]
    else:
        constraint_index = descriptor["constraint"]
        organized_column = flag_events[:, round_index, constraint_index]

    original_column = detector_events[:, detector_index]
    np.testing.assert_array_equal(organized_column, original_column)

print("All detector columns are preserved.")

### Check the noiseless case

The clean circuit should produce zero detector events and zero logical
flips. Its existing face checks must remain present in the mask.

In [ ]:
clean_sampler = clean_circuit.compile_sampler(seed=seed)
clean_measurements = clean_sampler.sample(shot_count)

clean_events, clean_logical_flips = learning.measurement_parities(
    clean_circuit,
    clean_measurements,
    reference_measurements,
    descriptors,
)

has_clean_events = np.any(clean_events)
has_clean_logical_flips = np.any(clean_logical_flips)
assert not has_clean_events
assert not has_clean_logical_flips

clean_inputs = learning.structure_detectors(
    clean_events,
    descriptors,
    rounds,
    face_count,
    constraint_count,
)
clean_presence = clean_inputs.face_present
np.testing.assert_array_equal(clean_presence, face_present)

print("Noiseless events, logical flips, and presence checks passed.")

## Try things out

- Change the mask and predict its selected positions.
- Change `basis` to `"X"` or `start_phase` to `1`, then restart and run all cells.
- Try distance 7 with three rounds before increasing the shot count.
- Set the noise probability to zero and inspect the event arrays.

Exercise: find the first face detector, then print the face-event array for
that detector's shot, basis, round, and face. Use the scaffold below.
Only one basis appears in the first round; presence tells you which checks
actually exist. Optional extension: compare the event counts for two noise
probabilities while keeping the circuit size fixed. Small sample counts
are for exploration and do not establish decoder performance.

In [ ]:
first_face_descriptor = None
for descriptor in descriptors:
    if descriptor["kind"] == "face":
        first_face_descriptor = descriptor
        break

pprint(first_face_descriptor)
# Try indexing face_events using the values in this description.

## 5. Generate adapted local correction targets

The new package tracks boundary frames, removes Noah's known byproducts,
and produces spatial and temporal correction labels. Six input channels
retain the flag information. Four output channels describe the corrections.
Read `training_data/planar_488/README.md` for the exact label definitions.
This cell uses NumPy and Stim, so it works with the existing notebook kernel.

In [ ]:
from training_data.planar_488.generator import generate_batch as generate_training_batch

training_seed = seed + 1
training_batch = generate_training_batch(
    distance=distance,
    rounds=rounds,
    basis=basis,
    start_phase=start_phase,
    probability=noise_probability,
    shots=8,
    seed=training_seed,
)

print("Training inputs:", training_batch.inputs.shape)
print("Correction targets:", training_batch.labels.shape)
print("Loss mask:", training_batch.target_mask.shape)
print("All target parity and logical checks passed.")

### Inspect one boundary's labels

The arrays below use `(X/Z, data qubit)` ordering. Reduction may change the
representative while preserving the syndrome and logical class. The model's
first two output channels reverse that component order to Z, X.
Temporal targets have `(X/Z check, face)` ordering.

The prepared `.venv` includes Torch. Submit `sbatch scripts/smoke_test.sbatch`
from the repository root for the GPU optimizer, checkpoint, and cleanup check.
Select the `.venv` environment for this notebook in VS Code.

In [ ]:
shot_index = 0
boundary_index = 0
raw_errors = training_batch.corrections.raw_spatial[shot_index, boundary_index]
reduced_errors = training_batch.corrections.spatial[shot_index, boundary_index]
temporal_errors = training_batch.corrections.temporal[shot_index, boundary_index]

print("Raw spatial errors:")
print(raw_errors)
print("Reduced spatial errors:")
print(reduced_errors)
print("Temporal residuals:")
print(temporal_errors)